In [5]:
import os
import re
import json
import math

from IPython.display import JSON

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from google import genai
from google.genai import types
from ydata_profiling import ProfileReport

# Loading our python file

from scripts import cleaner
from scripts import runner

from scripts import report_compressor as rc

In [6]:
# client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

### Step 1: Loading the dataset and Metadata

In [7]:

# df = pd.read_csv(r"C:\Users\shubh\Desktop\Auto EDA\notebooks\Zomato Restaurant Dataset.csv")

df = pd.read_csv(r"C:\Users\shubh\Desktop\Auto EDA\datasets\harry_potter_school_log_messy.csv")

def get_dataframe_metadata(df):

    metadata = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "columns": {}
    }

    for column in df.columns:

        metadata["columns"][column] = {
            "dtype": str(df[column].dtype),
            "missing_count": int(df[column].isna().sum()),
            "unique_count": int(df[column].nunique()),
            "sample_values": (
                df[column]
                .dropna()
                .head(5)
                .tolist()
            )
        }

    return metadata

metadata = get_dataframe_metadata(df)

### Step 2: Generates Cleaning Step from Y-Profiling Report (somewhere else)

In [8]:

# calls cleaner.py 

json_report = r"C:\Users\shubh\Desktop\Auto EDA\JSON reports\compressed_report.json"

cleaning_steps = cleaner.get_cleaning_recommendations(json_report, df)

print(cleaning_steps)

Based on the provided reports, here is the summary of data cleaning steps required:

### 1. Duplicate Rows
* **Column:** Entire Dataset
* **Issue:** 6 duplicate rows identified.
* **Action:** Drop duplicate rows using `drop_duplicates()`.

### 2. Missing Values
* **Columns:** `Student_Name`, `House`, `Subject`, `Class_Date`, `Score`, `Attendance_Status`, `Teacher`
* **Issue:** Missing values are present across multiple columns (ranging from 1 to 6 missing values).
* **Action:** Handle missing values by imputation (e.g., filling categorical columns with mode or "Unknown") or dropping rows depending on analysis requirements.

### 3. Incorrect Data Types
* **Column:** `Score`, `Year_Level`, `Class_Date`
* **Issue:** Stored as text (object) type instead of numeric or datetime formats.
* **Action:** Convert `Score` and `Year_Level` into numeric format (ignoring text/invalid entries that result in NaNs), and convert `Class_Date` to datetime format.

### 4. Outliers, Suspicious Values, and Fo

### Step 3: Takes Cleaning Steps and Generates Python Code for Them

In [9]:

# stores generated python code for cleaning functions in a variable cleaning_functions_json

cleaning_functions_json = cleaner.generate_cleaning_functions(cleaning_steps, metadata)

print(cleaning_functions_json)

{
    "functions": [
        {
            "name": "clean_duplicates",
            "columns": ["Entire Dataset"],
            "action": "drop_duplicates",
            "reason": "Drop exact duplicate rows from the dataset.",
            "requires_review": false,
            "code": "def clean_duplicates(df):\n    before = len(df)\n    df_clean = df.drop_duplicates(keep='first').reset_index(drop=True)\n    print(f'Removed {before - len(df_clean)} duplicate rows')\n    return df_clean"
        },
        {
            "name": "clean_categorical_values",
            "columns": ["Student_Name", "House", "Subject", "Attendance_Status", "Teacher"],
            "action": "categorical_cleanup",
            "reason": "Standardize whitespace, case, and synonyms for categorical columns.",
            "requires_review": false,
            "code": "def clean_categorical_values(df):\n    df_clean = df.copy()\n    def clean_text(series):\n        return (series.astype('string').str.strip().str.replace

### Step 4: Runs the Generates Function using Runner.py


In [10]:

# runs cleaning functions and returns cleaned dataframe

cleaned_df = runner.execute_steps(cleaning_functions_json, df, metadata) 

Removed 6 duplicate rows
[SUCCESS] clean_duplicates ran successfully.
[SUCCESS] clean_categorical_values ran successfully.
[SUCCESS] convert_numeric_columns ran successfully.
[SUCCESS] parse_class_date ran successfully.
Score: 6 out-of-range values set to NaN
[SUCCESS] handle_outliers ran successfully.
[SUCCESS] handle_missing_values ran successfully.
Cleaning process completed.


In [11]:
cleaned_df.head(5)

cleaned_df.to_csv("cleaned_data.csv", index=False)

### Step 5: Testing code-repair function by passing error functions

In [ ]:
error_function_json = r"""{
    "functions": [
        {
            "name": "remove_duplicate_rows",
            "columns": [],
            "action": "drop_duplicates",
            "reason": "The dataset contains 6 duplicate rows which should be removed.",
            "requires_review": false,
            "code": "def remove_duplicate_rows(df):\n    return df.drop_duplicate()"
        },
        {
            "name": "clean_student_name_whitespace",
            "columns": ["Student_Name"],
            "action": "strip_whitespace",
            "reason": "Student_Name contains leading whitespace.",
            "requires_review": false,
            "code": "def clean_student_name_whitespace(df):\n    df = df.copy()\n    if 'Student_Name' in df.columns:\n        df['Student_Name'] = df['StudentName'].str.strip()\n    return df"
        },
        {
            "name": "clean_score_formatting",
            "columns": ["Score"],
            "action": "clean_and_cast_numeric",
            "reason": "Score is read as text, contains formatting anomalies like trailing .0.0, and invalid out-of-range values.",
            "requires_review": true,
            "code": "import numpy as np\ndef clean_score_formatting(df):\n    df = df.copy()\n    if 'Score' in df.columns:\n        df['Score'] = df['Score'].astype(str).str.replace(r'\\.0+', '', regex=True)\n        df['Score'] = pd.to_numeric(df['Score'], errors='coerce')\n        df.loc[(df['Score'] < 0) | (df['Score'] > 100), 'Score'] = np.nan"
        },
        {
            "name": "standardize_class_date",
            "columns": ["Class_Date"],
            "action": "standardize_dates",
            "reason": "Class_Date uses mixed formats (dashes vs slashes) and contains text/invalid entries.",
            "requires_review": true,
            "code": "def standardize_class_date(df):\n    df = df.copy()\n    if 'Class_Date' in df.columns:\n    df['Class_Date'] = pd.to_datetime(df['Class_Date'].str.replace('/', '-', regex=False), errors='coerce')\n    return df"
        },
        {
            "name": "standardize_subject",
            "columns": ["Subject"],
            "action": "standardize_text",
            "reason": "Subject contains case inconsistencies, trailing spaces, and abbreviations.",
            "requires_review": true,
            "code": "def standardize_subject(df):\n    df = df.copy()\n    if 'Subject' in df.columns:\n        df['Subject'] = df['Subject'].str.strip().str.title()\n        df['Subject'] = df['Subject'].replace(subject_map)\n    return df"
        },
        {
            "name": "standardize_attendance_status",
            "columns": ["Attendance_Status"],
            "action": "standardize_categorical",
            "reason": "Attendance_Status contains inconsistent casing and alternative values like 'here'.",
            "requires_review": true,
            "code": "def standardize_attendance_status(df, mode):\n    df = df.copy()\n    if 'Attendance_Status' in df.columns:\n        df['Attendance_Status'] = df['Attendance_Status'].str.strip().str.title()\n        df['Attendance_Status'] = df['Attendance_Status'].replace({'Here': 'Present'})\n    return df"
        },
        {
            "name": "standardize_teacher",
            "columns": ["Teacher"],
            "action": "standardize_text",
            "reason": "Teacher has mixed casing across records.",
            "requires_review": false,
            "code": "def standardise_teacher(df):\n    df = df.copy()\n    if 'Teacher' in df.columns:\n        df['Teacher'] = df['Teacher'].str.strip().str.title()\n    return df"
        }
    ]
}"""

runner.execute_steps(error_function_json, df, metadata)

[FAILED] remove_duplicate_rows — AttributeError: 'DataFrame' object has no attribute 'drop_duplicate'
Calling Code Repair function to fix the error...
